# Structured Data Handling: Tables → LLMs
**Notebook author: Dr. Monali Mavani**   |  **AIML ZG536 · LLMs for Generative AI**

**What you'll do in this lab**

1. Understand the *three paths* by which a table can reach an LLM, and when to pick which.
2. See all three run against the same small, hand-verifiable table.
3. Zoom in on path (a) to compare Markdown / CSV / JSON serializations.
4. Apply the paths to **real sources** — a CSV, PDFs (both text-selectable and image-only).

## Learning objectives

By the end of this notebook you will be able to:

1. **Name the three paths** by which a table can reach an LLM — serialize (text tokens), image
   (vision-language model), and query (model writes code, Python executes it) — and state what each
   puts on the model vs. on the host.
2. **Run all three paths on the same small table** and grade them against a hand-computed ground truth,
   so the comparison is controlled and verifiable.
3. **Compare serialization formats** (Markdown, CSV, JSON) on token cost and accuracy, and explain why
   no single format wins universally.
4. **Apply the paths to real sources** — a CSV file, a text-selectable PDF, and an image-only
   (scanned) PDF — and match each source to its natural path.
5. **Explain why path (c) (query) shifts risk** from arithmetic errors to semantic errors — the code
   Python runs is guaranteed correct, but it may answer the *wrong* question.


## Before you run — one-time setup

This notebook runs on **Colab** or **Kubeflow** and **requires a GPU** (the vision-language model is
too slow on CPU). The first code cell asserts a GPU is present.

### On Kubeflow
1. Request a **GPU** pod / notebook server.
2. Run `bash setup_env.sh` once in a terminal (creates the **Python (LLM venv)** kernel).
3. Keep `labkit.py` in the **same `~/data` folder** as this notebook.
4. Kernel → Change Kernel → **Python (LLM venv)**, then run all cells.

### On Colab
1. Runtime → Change runtime type → **GPU** (required).
2. Put `labkit.py` in your Google Drive **"Colab Notebooks"** folder (once).
3. Run the first cell and **approve the Drive-access popup**.
4. Models **download fresh each session** — no Drive space used.


In [3]:
# --- Portable bootstrap: Colab or Kubeflow ---
import importlib.util, sys
if importlib.util.find_spec("google.colab") is not None:
    from google.colab import drive
    drive.mount("/content/drive")
    sys.path.append("/content/drive/MyDrive/Colab Notebooks")

import labkit
L = labkit.setup()

assert L.IS_GPU, (
    "This lab requires a GPU. On Kubeflow request a GPU pod; on Colab set "
    "Runtime -> Change runtime type -> GPU, then re-run from the top."
)

labkit.ensure({
    "transformers": "transformers>=4.44,<5",
    "accelerate":   "accelerate>=0.33",
    "pdfplumber":   "pdfplumber",
    "tabulate":     "tabulate",
    "qwen_vl_utils":"qwen-vl-utils",
})


env=local  device=cpu  dtype=float32  gpu=none
persistent root : C:\Users\pc/lab-data
HF cache        : C:\Users\pc/lab-data\hf-cache


AssertionError: This lab requires a GPU. On Kubeflow request a GPU pod; on Colab set Runtime -> Change runtime type -> GPU, then re-run from the top.

## Troubleshooting

If a notebook won't start, a model won't load, or you hit "CUDA out of memory" / "disk is full":
**open a terminal and follow `DIAGNOSTICS.md`**. The cell below is a *mid-lab* convenience only.


In [ ]:
import torch
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM free/total: {free/1e9:.1f}/{total/1e9:.1f} GB")
else:
    print("No GPU visible — see DIAGNOSTICS.md.")
print("HF cache:", L.CACHE_ROOT)


GPU: NVIDIA A100-SXM4-80GB | VRAM free/total: 69.4/85.0 GB
HF cache: /home/jovyan/data/hf-cache


## Load the text model and define `ask()`

Everything except path (b) uses this Qwen instruction-tuned model. `ask()` is the
same minimal text-in / text-out helper used across earlier labs.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub.utils import LocalEntryNotFoundError

TEXT_MODEL = "Qwen/Qwen2.5-3B-Instruct"

model_kwargs = dict(
    torch_dtype=getattr(torch, L.DTYPE),
    device_map="auto",
    low_cpu_mem_usage=True,
)

print(f"Searching local cache for {TEXT_MODEL} ...")
try:
    tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL, local_files_only=True)
    model = AutoModelForCausalLM.from_pretrained(TEXT_MODEL, local_files_only=True, **model_kwargs)
    print("Loaded from local cache!")
except (LocalEntryNotFoundError, EnvironmentError, OSError):
    print("Not in cache — downloading ...")
    tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL, local_files_only=False)
    model = AutoModelForCausalLM.from_pretrained(TEXT_MODEL, local_files_only=False, **model_kwargs)
    print("Download complete, cached for next time.")

model.eval()
print("Text model ready.")


Searching local cache for Qwen/Qwen2.5-3B-Instruct ...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded from local cache!
Text model ready.


In [ ]:
def ask(messages, max_tokens=200, temperature=0.0):
    """Text -> text. Deterministic by default."""
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    gen_kwargs = dict(max_new_tokens=max_tokens,
                      do_sample=temperature > 0,
                      pad_token_id=tokenizer.eos_token_id)
    if temperature > 0:
        gen_kwargs["temperature"] = temperature
    with torch.no_grad():
        out = model.generate(**inputs, **gen_kwargs)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


---
# The Concept — Three Paths for a Table

A table doesn't have one obvious way to reach an LLM. There are three
fundamentally different **interfaces** between structured data and a model.

|                   | Who reads the values?     | Who does the arithmetic? |
|-------------------|---------------------------|--------------------------|
| **(a) Serialize** | LLM (as text tokens)      | LLM                      |
| **(b) Image**     | LLM (as image tokens)     | LLM                      |
| **(c) Query**     | Python / SQL              | Python / SQL             |

- **Path (a) — Serialize.** Turn the table into text (Markdown, CSV, JSON) and paste
  it into the prompt. Model reads *and* computes.
- **Path (b) — Image.** Feed the pixels to a vision-language model. Model *sees* the
  numbers, reasons about what it sees. No text conversion at all.
- **Path (c) — Query.** Give the model only the *schema* and a couple of sample rows;
  it writes a short pandas expression or SQL statement; Python executes it.

Paths (a) and (b) put both extraction and computation on the model. Path (c) splits
the burden — the model translates the question into code; Python does the arithmetic
it can't get wrong.

The choice isn't "which is best" — it's "which fits **this** source and **this**
question." All three appear in real systems, often side by side.

## The demo table + generate sample source files

We use a 4-row table small enough to verify answers by hand. Below we write it out
in every format Parts A–C will need: CSV, SQLite, PDF (text-selectable), and PNG
(image-only, standing in for a scanned PDF page).

In [5]:
import pandas as pd, sqlite3, matplotlib.pyplot as plt

table_data = [
    {"region": "North", "product": "Widget A", "units_sold": 120, "revenue": 2400},
    {"region": "South", "product": "Widget B", "units_sold":  85, "revenue": 2125},
    {"region": "East",  "product": "Widget A", "units_sold":  60, "revenue": 1200},
    {"region": "West",  "product": "Widget C", "units_sold": 200, "revenue": 5000},
]
df = pd.DataFrame(table_data)

# Ground truth -- computed directly, used to grade every path
GT_TOTAL_REVENUE = sum(r["revenue"] for r in table_data)      # 10725
GT_TOP_REGION    = max(table_data, key=lambda r: r["revenue"])["region"]   # West
print(f"Ground truth: total revenue = {GT_TOTAL_REVENUE}, top region = {GT_TOP_REGION}")

# Write the same data to every source format we'll need downstream
df.to_csv("sales.csv", index=False)

# Render as PDF (vector: text stays extractable) and as PNG (image only)
fig, ax = plt.subplots(figsize=(6, 2)); ax.axis("off")
tbl = ax.table(cellText=df.values.tolist(), colLabels=df.columns.tolist(),
               loc="center", cellLoc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1, 1.8)
plt.tight_layout()
plt.savefig("sales.pdf", bbox_inches="tight")
plt.savefig("sales_image.png", dpi=150, bbox_inches="tight")
plt.close()

print("Wrote: sales.csv, sales.pdf, sales_image.png")

Ground truth: total revenue = 10725, top region = West
Wrote: sales.csv, sales.pdf, sales_image.png


In [6]:
# The single question we ask against every path and every source
question = ("What is the total revenue across all regions, "
            "and which single region contributed the most?")

def check(text_answer):
    """Grade an answer against ground truth. Returns (total_ok, region_ok)."""
    s = str(text_answer)
    return (str(GT_TOTAL_REVENUE) in s,
            GT_TOP_REGION.lower() in s.lower())

---
# Part A — The three paths, on one small table

Each path answers the *same* question about the *same* data. We grade all three
against the ground truth computed above.

## A(a) — Serialize into the prompt

The table is turned into Markdown text and pasted into the message. The model reads
the numbers as tokens.

In [7]:
markdown_table = df.to_markdown(index=False)
print(markdown_table)

| region   | product   |   units_sold |   revenue |
|:---------|:----------|-------------:|----------:|
| North    | Widget A  |          120 |      2400 |
| South    | Widget B  |           85 |      2125 |
| East     | Widget A  |           60 |      1200 |
| West     | Widget C  |          200 |      5000 |


In [8]:
prompt = f"Here is a table in Markdown format:\n\n{markdown_table}\n\nQuestion: {question}"
path_a_answer = ask([{"role": "user", "content": prompt}])
print(path_a_answer)

/home/jovyan/data/venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/jovyan/data/venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/jovyan/data/venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


To answer your question, let's first calculate the total revenue across all regions.

The total revenue can be calculated by summing up the revenue values from each row:
- North: $2400$
- South: $2125$
- East: $1200$
- West: $5000$

Total revenue = $2400 + 2125 + 1200 + 5000 = 10725$

So, the total revenue across all regions is $10725.

Next, to determine which single region contributed the most to this total revenue, we need to look at the individual revenue contributions for each region:
- North: $2400$
- South: $2125$
- East: $1200$
- West: $5000$

The highest revenue among these is from the West region with $5000.




## A(b) — Show it as an image

We hand the rendered PNG to a vision-language model. **This loads `Qwen2.5-VL`,
a second model family, so the first run downloads more weights.**

We wrap the boilerplate in a `vl_ask()` helper so the call site mirrors `ask()`.

In [10]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from huggingface_hub.utils import LocalEntryNotFoundError
from qwen_vl_utils import process_vision_info

VL_MODEL = "Qwen/Qwen2-VL-2B-Instruct"   # Qwen2-VL works with transformers 4.46.3 (Qwen2.5-VL needs >= 4.49)

print(f"Searching local cache for {VL_MODEL} ...")
try:
    vl_processor = AutoProcessor.from_pretrained(VL_MODEL, local_files_only=True)
    vl_model = Qwen2VLForConditionalGeneration.from_pretrained(
        VL_MODEL, torch_dtype=getattr(torch, L.DTYPE), device_map="auto", local_files_only=True)
    print("Vision-language model loaded from cache!")
except (LocalEntryNotFoundError, EnvironmentError, OSError):
    print("Not in cache — downloading ...")
    vl_processor = AutoProcessor.from_pretrained(VL_MODEL, local_files_only=False)
    vl_model = Qwen2VLForConditionalGeneration.from_pretrained(
        VL_MODEL, torch_dtype=getattr(torch, L.DTYPE), device_map="auto", local_files_only=False)
    print("Download complete, cached for next time.")

Searching local cache for Qwen/Qwen2-VL-2B-Instruct ...
Not in cache — downloading ...


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/429M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

Download complete, cached for next time.


In [11]:
def vl_ask(image_path: str, question: str, max_tokens: int = 200) -> str:
    """Vision counterpart to ask(): image + text in, string out."""
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image_path},
        {"type": "text",  "text":  question},
    ]}]
    text = vl_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(messages)
    inputs = vl_processor(text=[text], images=image_inputs,
                          padding=True, return_tensors="pt").to(vl_model.device)
    with torch.no_grad():
        out = vl_model.generate(**inputs, max_new_tokens=max_tokens)
    return vl_processor.decode(out[0][inputs.input_ids.shape[1]:],
                               skip_special_tokens=True)

path_b_answer = vl_ask("sales_image.png", question)
print(path_b_answer)

To find the total revenue across all regions, we add up the revenue for each region:

- North: 2400
- South: 2125
- East: 1200
- West: 5000

Total revenue = 2400 + 2125 + 1200 + 5000 = 12625

The single region that contributed the most revenue is the North region, with a total revenue of $2400.


## A(c) — Query via pandas

The model sees only the *schema* and two sample rows — never the values. It writes
one pandas expression; Python executes it in a locked-down namespace.

The `safe_exec_query` helper is the pedagogical core of this path: sandboxed
`__builtins__`, disallow-list of dangerous patterns, fenced-code extraction. Same
primitives as the AST-safe eval in the ReAct lab.

In [12]:
import re

def extract_code(text: str) -> str:
    m = re.search(r"```(?:python)?\n?(.*?)```", text, re.DOTALL)
    return m.group(1).strip() if m else text.strip()

FORBIDDEN_PY = ["import", "open(", "exec(", "eval(", "os.", "sys.", "__", "subprocess"]

def safe_exec_query(code_text: str, df: pd.DataFrame):
    """Sandboxed pandas exec. Returns (result, cleaned_code_or_error_message)."""
    code = extract_code(code_text)
    if any(bad in code for bad in FORBIDDEN_PY):
        return None, f"Rejected -- disallowed pattern:\n{code}"
    ns = {"df": df, "pd": pd, "__builtins__": {}}
    try:
        exec(code, ns)
        return ns.get("result"), code
    except Exception as e:
        return None, f"Execution error: {e}\nCode:\n{code}"


In [13]:
query_prompt = (
    f"You are given a pandas DataFrame `df` with columns and dtypes:\n"
    f"{df.dtypes.to_string()}\n\n"
    f"Sample rows:\n{df.head(2).to_string()}\n\n"
    "Write ONE short pandas expression that assigns the answer to a variable "
    "named `result`. Return ONLY the code, no explanation.\n\n"
    f"Question: {question}"
)
generated_code = ask([{"role": "user", "content": query_prompt}], max_tokens=150)
path_c_result, info = safe_exec_query(generated_code, df)
print(f"Executed:\n{info}\n\nResult: {path_c_result}")

/home/jovyan/data/venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/jovyan/data/venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/jovyan/data/venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


Executed:
result = df.groupby('region')['revenue'].sum().idxmax()

Result: West


## Compare all three

In [16]:
print(f"Ground truth: total revenue = {GT_TOTAL_REVENUE}, top region = {GT_TOP_REGION}\n")
print(f"{'Path':<18}{'Total ✓':<12}{'Region ✓':<12}{'Verdict'}")
print("-" * 52)

for name, ans in [("(a) Serialize", path_a_answer),
                  ("(b) VL image",  path_b_answer),
                  ("(c) DF query",  path_c_result)]:
    t_ok, r_ok = check(ans)
    verdict = "both correct" if t_ok and r_ok else (
              "partial"      if t_ok or r_ok else "both wrong")
    print(f"{name:<18}{str(t_ok):<12}{str(r_ok):<12}{verdict}")

print(f"""
What this tells you:
  (a) Serialize — the model read the table as text tokens and did the arithmetic
      itself. On a 4-row table it usually gets both right, but the addition is
      done by the LLM, not Python — it can drift on larger tables.

  (b) VL image  — the vision model read pixel values, not text. Accuracy depends
      on image resolution: at 150 dpi this table is legible, but lower dpi or
      messier layouts (real scanned documents) cause digit-reading errors.

  (c) DF query  — the model never saw the numbers. It wrote a pandas expression;
      Python executed it. The arithmetic is guaranteed correct — the only risk is
      whether the generated code answered the RIGHT question (semantic error, not
      arithmetic error). Check the printed code above to verify.
""")

Ground truth: total revenue = 10725, top region = West

Path              Total ✓     Region ✓    Verdict
----------------------------------------------------
(a) Serialize     True        True        both correct
(b) VL image      False       True        partial
(c) DF query      False       True        partial

What this tells you:
  (a) Serialize — the model read the table as text tokens and did the arithmetic
      itself. On a 4-row table it usually gets both right, but the addition is
      done by the LLM, not Python — it can drift on larger tables.

  (b) VL image  — the vision model read pixel values, not text. Accuracy depends
      on image resolution: at 150 dpi this table is legible, but lower dpi or
      messier layouts (real scanned documents) cause digit-reading errors.

  (c) DF query  — the model never saw the numbers. It wrote a pandas expression;
      Python executed it. The arithmetic is guaranteed correct — the only risk is
      whether the generated code answe

### A note on serialization format

Part A used Markdown, but you could also paste the table as CSV or JSON. The tradeoffs
in brief:

- **CSV** — cheapest on tokens (headers appear once, no decoration), but the model has
  to infer column-value alignment from position alone.
- **Markdown** — slightly more tokens (pipe characters, alignment row), but reads most
  naturally for instruct-tuned models trained on Markdown-heavy data.
- **JSON** — most expensive (every row repeats all key names), but each value is
  explicitly labelled, which helps on wide tables with many columns.

On a small table like this one, all three produce the same answer. The choice matters
at scale — hundreds of rows or dozens of columns — where token cost and positional
ambiguity start to bite. For most use cases, **Markdown or CSV** is the practical default.

---
# Part B — Real sources

The demo table lived in a Python `list[dict]`. Real tables live in files and
databases. Each source has a natural path — sometimes more than one.

## C.1 — CSV

In [19]:
csv_df = pd.read_csv("sales.csv")
print(csv_df, "\n")

prompt = (f"Here is a table in Markdown format:\n\n"
          f"{csv_df.to_markdown(index=False)}\n\nQuestion: {question}")
print(ask([{"role": "user", "content": prompt}]))

  region   product  units_sold  revenue
0  North  Widget A         120     2400
1  South  Widget B          85     2125
2   East  Widget A          60     1200
3   West  Widget C         200     5000 



/home/jovyan/data/venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/jovyan/data/venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/jovyan/data/venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


To answer your question, let's first calculate the total revenue across all regions.

The total revenue can be calculated by summing up the revenue values from each row:
- North: $2400$
- South: $2125$
- East: $1200$
- West: $5000$

Total revenue = $2400 + 2125 + 1200 + 5000 = 10725$

So, the total revenue across all regions is $10725.

Next, to determine which single region contributed the most to this total revenue, we need to look at the individual revenue contributions for each region:
- North: $2400$
- South: $2125$
- East: $1200$
- West: $5000$

The highest revenue among these is from the West region with $5000.




## C.2 — PDF, text-selectable (born-digital)

`sales.pdf` was rendered from matplotlib, so the text is still selectable. `pdfplumber`
pulls the table out row by row; from there it's a normal DataFrame — path (a) or (c)
works, same as C.1.

In [20]:
import pdfplumber

with pdfplumber.open("sales.pdf") as pdf:
    raw = pdf.pages[0].extract_table()

header, *rows = raw
pdf_df = pd.DataFrame(rows, columns=header)
pdf_df["units_sold"] = pdf_df["units_sold"].astype(int)
pdf_df["revenue"]    = pdf_df["revenue"].astype(int)
print(pdf_df, "\n")

prompt = (f"Here is a table in Markdown format:\n\n"
          f"{pdf_df.to_markdown(index=False)}\n\nQuestion: {question}")
print(ask([{"role": "user", "content": prompt}]))

  region   product  units_sold  revenue
0  North  Widget A         120     2400
1  South  Widget B          85     2125
2   East  Widget A          60     1200
3   West  Widget C         200     5000 



/home/jovyan/data/venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/jovyan/data/venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/jovyan/data/venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


To answer your question, let's first calculate the total revenue across all regions.

The total revenue can be calculated by summing up the revenue values from each row:
- North: $2400$
- South: $2125$
- East: $1200$
- West: $5000$

Total revenue = $2400 + 2125 + 1200 + 5000 = 10725$

So, the total revenue across all regions is $10725.

Next, to determine which single region contributed the most to this total revenue, we need to look at the individual revenue contributions for each region:
- North: $2400$
- South: $2125$
- East: $1200$
- West: $5000$

The highest revenue among these is from the West region with $5000.




## C.3 — PDF, image-only (scanned)

If `pdfplumber.extract_table()` had returned `None` (as it would for a scanned page),
we couldn't build a DataFrame. Path (b) skips the extraction step entirely — hand
the page image straight to the vision-language model.

In [21]:
# In practice you'd use pdf2image to rasterize a scanned page.
# For this demo, sales_image.png already stands in for the rasterized page.
print(vl_ask("sales_image.png", question))

To find the total revenue across all regions, we add up the revenue for each region:

- North: 2400
- South: 2125
- East: 1200
- West: 5000

Total revenue = 2400 + 2125 + 1200 + 5000 = 12625

The single region that contributed the most revenue is the North region, with a total revenue of $2400.


## Practice

1. **Add a 5th row** to `table_data` (e.g. `{"region": "Central", "product": "Widget A",
   "units_sold": 150, "revenue": 3750}`), recompute `GT_TOTAL_REVENUE` and `GT_TOP_REGION` by
   hand, then re-run all three paths. Does path (a) still get the arithmetic right on a 5-row
   table? At how many rows would you stop trusting the model's addition?

2. **Break path (b) on purpose.** Re-render `sales_image.png` at `dpi=50` (replace `dpi=150` in
   the generation cell) and re-run the VL path. At what DPI does the vision model start misreading
   digits? What does this say about using path (b) on real scanned documents?

3. **Make path (c) answer the wrong question.** Change the question to something ambiguous, like
   `"Which product did best?"` (does "best" mean highest revenue or most units sold?). Run path (c)
   and read the generated code — which interpretation did the model pick? Try the same ambiguous
   question on path (a) — does it pick the same interpretation? What does the difference tell you
   about where each path's risk sits?


## Disk footprint

Two models download to the cache; the lab writes no training output.
Read-only usage check against a 15 GB budget:


In [ ]:
labkit.disk_report(budget_gb=15)   # read-only; deletes nothing
